In [3]:
import sys
sys.path.append('../')

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from helpers.cm26 import DatasetCM26, read_datasets
from helpers.selectors import *
from helpers.operators import Coarsen, Filtering, Subsampling, CoarsenKochkov, CoarsenWeighted, CoarsenKochkovMinMax
from helpers.state_functions import *

import hvplot
import cmocean

%load_ext autoreload
%autoreload 3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [108]:
ds = read_datasets(['validate'], [15], 'subfilter-large')['validate-15']

Reading from folder /vast/pp2681/CM26_datasets/ocean3d/subfilter-large/FGR3/factor-15


In [109]:
ds2d = ds.select2d(time=5, zl=5)

In [110]:
EXP1 = import_ANN('/scratch/pp2681/mom6/CM26_ML_models/ocean3d/subfilter/FGR3/EXP1/model/Tall.nc')
ann1 = ds2d.state.Apply_ANN(ann_Tall=EXP1,gradient_features=['sh_xy', 'sh_xx', 'rel_vort'])

In [111]:
ann2 = ds.state.ANN_inference(ann_Tall=EXP1, gradient_features=['sh_xy', 'sh_xx', 'rel_vort'], zl=5, time=5)

In [112]:
def norm(x):
    return torch.max(torch.abs(x))

In [113]:
norm(ann1['Txx'] - ann2['Txx'])

tensor(1.1176e-08, grad_fn=<MaxBackward1>)

In [114]:
norm(ann1['Tyy'] - ann2['Tyy'])

tensor(1.1176e-08, grad_fn=<MaxBackward1>)

In [115]:
norm(ann1['Txy'] - ann2['Txy'])

tensor(3.8417e-09, grad_fn=<MaxBackward1>)

In [118]:
%time ann1 = ds2d.state.Apply_ANN(ann_Tall=EXP1,gradient_features=['sh_xy', 'sh_xx', 'rel_vort'])

CPU times: user 65.4 ms, sys: 10.9 ms, total: 76.3 ms
Wall time: 6 ms


In [119]:
%time ann2 = ds.state.ANN_inference(ann_Tall=EXP1, gradient_features=['sh_xy', 'sh_xx', 'rel_vort'], zl=5, time=5)

CPU times: user 168 ms, sys: 12.8 ms, total: 181 ms
Wall time: 14.2 ms


In [120]:
%%time
for zl in range(10):
    for time in range(12):
        ds2d = ds.select2d(time=time, zl=zl)
        ann1 = ds2d.state.Apply_ANN(ann_Tall=EXP1,gradient_features=['sh_xy', 'sh_xx', 'rel_vort'])

CPU times: user 1min 32s, sys: 344 ms, total: 1min 32s
Wall time: 27.5 s


In [121]:
%%time
for zl in range(10):
    for time in range(12):
        ann2 = ds.state.ANN_inference(ann_Tall=EXP1,gradient_features=['sh_xy', 'sh_xx', 'rel_vort'],zl=zl, time=time)

CPU times: user 18.1 s, sys: 65.5 ms, total: 18.2 s
Wall time: 1.31 s


In [122]:
%%time
ds2d = ds.select2d(time=0, zl=0)
for zl in range(10):
    for time in range(12):
        ann1 = ds2d.state.Apply_ANN(ann_Tall=EXP1,gradient_features=['sh_xy', 'sh_xx', 'rel_vort'])

CPU times: user 8.18 s, sys: 24.7 ms, total: 8.2 s
Wall time: 768 ms


In [123]:
%%time
for zl in range(10):
    for time in range(12):
        ann2 = ds.state.ANN_inference(ann_Tall=EXP1,gradient_features=['sh_xy', 'sh_xx', 'rel_vort'],zl=0, time=0)

CPU times: user 17.8 s, sys: 46.7 ms, total: 17.9 s
Wall time: 1.28 s


In [124]:
import itertools

In [172]:
rots  = [90, 0]
refxs = [True, False]
refys = [True, False]

In [182]:
%%time
ds2d = ds.select2d(time=np.random.randint(12), zl=np.random.randint(8))
for rotation, reflect_x, reflect_y in itertools.product(rots, refxs, refys):
    ann1 = ds2d.state.Apply_ANN(ann_Tall=EXP1,gradient_features=['sh_xy', 'sh_xx', 'rel_vort'], rotation=rotation, reflect_x=reflect_x, reflect_y=reflect_y)

CPU times: user 1.24 s, sys: 3.82 ms, total: 1.25 s
Wall time: 278 ms


In [183]:
%%time
time=np.random.randint(12); zl=np.random.randint(8)
ds2d = ds.select2d(time=np.random.randint(12), zl=np.random.randint(8))
for rotation, reflect_x, reflect_y in itertools.product(rots, refxs, refys):
    ann2 = ds.state.ANN_inference(ann_Tall=EXP1,gradient_features=['sh_xy', 'sh_xx', 'rel_vort'],
                                  zl=zl, time=time, rotation=rotation, reflect_x=reflect_x, reflect_y=reflect_y)

CPU times: user 1.5 s, sys: 37 ms, total: 1.54 s
Wall time: 134 ms
